<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-09/AI_Modul_9.5_Praktikum_Loss_Function.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 9.5: Praktikum Algoritma Forward Propagation & Loss Functions
## Mata Kuliah: Kecerdasan Buatan dan Sains Data Pertanian Presisi (INSTIPER)

---

### Tujuan Praktikum:
1. Membangun mekanisme perambatan maju (*Forward Propagation*) berbasis operasi matriks batch secara murni dengan NumPy.
2. Mengimplementasikan fungsi rugi *Mean Squared Error* (MSE), *Binary Cross-Entropy* (BCE), dan *Categorical Cross-Entropy* (CCE) yang stabil secara numerik.
3. Membuktikan keunggulan stabilitas numerik trik **Log-Sum-Exp** dalam mencegah *overflow* dan *underflow*.
4. Menerapkan fungsi rugi terbobot (*Weighted BCE*) pada studi kasus deteksi dini jamur busuk pangkal batang sawit (*Ganoderma boninense*).
5. Membuktikan secara komputasi mengapa fungsi rugi Cross-Entropy menghasilkan gradien yang jauh lebih responsif dibandingkan MSE pada luaran sigmoid.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Pengaturan visualisasi publikasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 120

print("[OK] Seluruh pustaka dasar Forward Propagation & Loss Functions berhasil dimuat!")

## 1. Implementasi Arsitektur Forward Propagation Tensorial
Kita membangun graf komputasi terarah (*Computational Graph*) yang menerima batch data masukan $\mathbf{X} \in \mathbb{R}^{B \times m}$ dan mengalirkannya melewati 2 lapisan tersembunyi beraktivasi ReLU menuju lapisan luaran Softmax.

In [ ]:
class BatchForwardMLP:
    """
    Graf komputasi perambatan maju multi-lapis berbasis batch matriks.
    """
    def __init__(self, layer_dims=[4, 8, 6, 3], seed=42):
        self.layer_dims = layer_dims
        self.params = {}
        rng = np.random.RandomState(seed)
        
        # Inisialisasi He Normal
        for l in range(1, len(layer_dims)):
            n_in = layer_dims[l-1]
            n_out = layer_dims[l]
            self.params[f'W{l}'] = rng.normal(0, np.sqrt(2.0 / n_in), size=(n_in, n_out))
            self.params[f'b{l}'] = np.zeros((1, n_out))
            
    def relu(self, Z):
        return np.maximum(0.0, Z)
        
    def softmax(self, Z):
        # Stabil secara numerik dengan pengurangan nilai maksimum
        exp_Z = np.exp(Z - np.max(Z, axis=1, keepdims=True))
        return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

    def forward(self, X):
        """
        Mengeksekusi perambatan maju dan menyimpan riwayat aktivasi (cache).
        """
        cache = {'A0': X}
        A = X
        L = len(self.layer_dims) - 1
        
        # Lapisan Tersembunyi (ReLU)
        for l in range(1, L):
            W = self.params[f'W{l}']
            b = self.params[f'b{l}']
            Z = np.dot(A, W) + b
            A = self.relu(Z)
            cache[f'Z{l}'] = Z
            cache[f'A{l}'] = A
            
        # Lapisan Luaran (Softmax)
        W_last = self.params[f'W{L}']
        b_last = self.params[f'b{L}']
        Z_last = np.dot(A, W_last) + b_last
        A_last = self.softmax(Z_last)
        cache[f'Z{L}'] = Z_last
        cache[f'A{L}'] = A_last
        
        return A_last, cache

print("[OK] Kelas BatchForwardMLP berhasil didefinisikan!")

## 2. Implementasi Fungsi-Fungsi Rugi (*Loss Functions*) Numerik Stabil
Di bawah ini kita mengimplementasikan:
1. `mean_squared_error`: Untuk target kontinu riil.
2. `binary_cross_entropy`: Untuk klasifikasi 2 kelas dengan pemotongan epsilon.
3. `categorical_cross_entropy`: Untuk klasifikasi multi-kelas one-hot.
4. `log_sum_exp_cce`: Evaluasi berbasis logit stabil tanpa ekspresi eksponensial eksplisit.

In [ ]:
def compute_mse(y_true, y_pred):
    return 0.5 * np.mean((y_true - y_pred)**2)

def compute_bce(y_true, y_pred, eps=1e-15):
    # Pemotongan epsilon untuk mencegah ln(0)
    y_pred_clipped = np.clip(y_pred, eps, 1.0 - eps)
    loss = -(y_true * np.log(y_pred_clipped) + (1.0 - y_true) * np.log(1.0 - y_pred_clipped))
    return np.mean(loss)

def compute_cce(y_true_onehot, y_pred_probs, eps=1e-15):
    y_pred_clipped = np.clip(y_pred_probs, eps, 1.0 - eps)
    loss = -np.sum(y_true_onehot * np.log(y_pred_clipped), axis=1)
    return np.mean(loss)

def compute_log_sum_exp_cce(Z_last, y_true_onehot):
    # Log-Sum-Exp Trick: log(sum(exp(Z))) = c + log(sum(exp(Z - c)))
    c = np.max(Z_last, axis=1, keepdims=True)
    log_sum_exp = c + np.log(np.sum(np.exp(Z_last - c), axis=1, keepdims=True))
    log_probs = Z_last - log_sum_exp
    loss = -np.sum(y_true_onehot * log_probs, axis=1)
    return np.mean(loss)

print("[OK] Seluruh fungsi rugi numerik stabil berhasil didefinisikan!")

## 3. Eksperimen Stabilitas Numerik: Mengapa Log-Sum-Exp Sangat Penting?
Mari kita uji apa yang terjadi pada komputasi naive saat dihadapkan pada nilai logit yang sangat besar ($z = 1000.0$) seperti yang kerap muncul pada jaringan dalam sebelum konvergen.

In [ ]:
logits_extreme = np.array([[1000.0, 1002.0, 998.0]])
target_onehot = np.array([[0, 1, 0]])

# Pendekatan Naive
try:
    exp_naive = np.exp(logits_extreme)
    probs_naive = exp_naive / np.sum(exp_naive, axis=1, keepdims=True)
    loss_naive = -np.sum(target_onehot * np.log(probs_naive))
    print(f"Hasil Naive Loss : {loss_naive}")
except Exception as e:
    print(f"Hasil Naive Gagal : {e}")

# Pendekatan Stabil Log-Sum-Exp
loss_stable = compute_log_sum_exp_cce(logits_extreme, target_onehot)
print(f"Hasil Stabil Log-Sum-Exp Loss : {loss_stable:.6f}")
print("Catatan: Komputasi stabil menghasilkan angka presisi tanpa overflow inf/NaN!")

## 4. Studi Kasus Agro-Industri 1: Deteksi Dini Jamur Ganoderma Boninense
Jamur *Ganoderma boninense* memicu kerugian ratusan juta rupiah. Kita menyimulasikan batch 100 pohon sawit:
- Target $1$: Pohon Terinfeksi Ganoderma ($5\%$ kasus langka).
- Target $0$: Pohon Bebas Jamur ($95\%$ kasus normal).

In [ ]:
np.random.seed(42)
n_samples = 100

# 95 pohon sehat, 5 pohon terinfeksi
y_ganoderma = np.zeros(n_samples)
y_ganoderma[:5] = 1.0  # 5 pohon terinfeksi

# Prediksi model probabilistik
y_pred_prob = np.random.uniform(0.01, 0.20, n_samples)
# Buat prediksi model pada pohon sakit bervariasi
y_pred_prob[0] = 0.92  # Terdeteksi sangat tepat
y_pred_prob[1] = 0.78  # Terdeteksi cukup tepat
y_pred_prob[2] = 0.35  # Ragu-ragu
y_pred_prob[3] = 0.05  # False Negative Parah!
y_pred_prob[4] = 0.01  # False Negative Sangat Fatal!

# Evaluasi Loss Standar vs Weighted BCE
bce_standar = compute_bce(y_ganoderma, y_pred_prob)

# Weighted BCE (Bobot penalti positif w_pos = 10.0 karena pohon sakit sangat fatal)
eps = 1e-15
p_clip = np.clip(y_pred_prob, eps, 1.0 - eps)
loss_weighted = -(10.0 * y_ganoderma * np.log(p_clip) + (1.0 - y_ganoderma) * np.log(1.0 - p_clip))
bce_weighted = np.mean(loss_weighted)

print("=== HASIL EVALUASI DETEKSI GANODERMA ===")
print(f"Nilai Kerugian Standar BCE : {bce_standar:.4f}")
print(f"Nilai Kerugian Weighted BCE: {bce_weighted:.4f}")
print(f"Lonjakan Penalti Asimetris : {bce_weighted / bce_standar:.2f}x lipat lebih keras terhadap False Negative!")

## 5. Visualisasi Profil Penalti: Mengapa Cross-Entropy Lebih Responsif Dibanding MSE?
Kita membandingkan respon nilai rugi dan magnitudo gradien pembaruan bobot terhadap logit ketika target sejati adalah $y=1$ (Pohon Sakit).

In [ ]:
z_arr = np.linspace(-4.5, 4.5, 300)
p_arr = 1.0 / (1.0 + np.exp(-z_arr))

# Target sejati y = 1
# Gradien Loss terhadap logit z:
# BCE: dL/dz = p - 1
# MSE: dL/dz = (p - 1) * p * (1 - p)
grad_bce_mag = np.abs(p_arr - 1.0)
grad_mse_mag = np.abs((p_arr - 1.0) * p_arr * (1.0 - p_arr))

plt.figure(figsize=(9, 5))
plt.plot(z_arr, grad_bce_mag, label=r'Magnitudo Gradien BCE: $|p - 1|$', color='#e74c3c', lw=2.5)
plt.plot(z_arr, grad_mse_mag, label=r'Magnitudo Gradien MSE: $|(p - 1)p(1 - p)|$', color='#2980b9', lw=2.5, linestyle='--')

plt.title('Komparasi Kekuatan Gradien Pembaruan Bobot Terhadap Logit ($z$)', fontweight='bold', pad=12)
plt.xlabel('Nilai Logit Masukan ($z$)')
plt.ylabel('Kekuatan Sinyal Gradien $|dL/dz|$')
plt.axvline(0, color='gray', linestyle=':', alpha=0.5)
plt.legend(fontsize=10)

plt.annotate('Zona Kelumpuhan Belajar MSE\n(Gradien mendekati 0 saat salah besar!)',
             xy=(-3.0, 0.05), xytext=(-4.0, 0.35),
             arrowprops=dict(facecolor='#2980b9', shrink=0.05, width=1.5, headwidth=7),
             fontweight='bold', color='#2471a3', fontsize=9)

plt.annotate('BCE Responsif Maksimal\n(Gradien mendekati 1.0 saat salah)',
             xy=(-3.0, 0.95), xytext=(-2.0, 0.70),
             arrowprops=dict(facecolor='#c0392b', shrink=0.05, width=1.5, headwidth=7),
             fontweight='bold', color='#c0392b', fontsize=9)

plt.show()

## 6. Studi Kasus Agro-Industri 2: Eksekusi Forward Pass Penuh pada 3 Kelas Mutu TBS
Kita membangkitkan batch 5 sampel TBS dengan 4 fitur optik, menjalankannya pada `BatchForwardMLP`, dan menghitung nilai kerugian CCE.

In [ ]:
# Masukan 5 sampel TBS x 4 fitur
X_batch = np.array([
    [0.18, 0.62, 0.12, 0.00],  # Sampel 1: Mentah
    [0.35, 0.48, 0.30, 0.05],  # Sampel 2: Kurang Matang
    [0.55, 0.30, 0.65, 0.20],  # Sampel 3: Matang Sempurna
    [0.58, 0.28, 0.70, 0.22],  # Sampel 4: Matang Sempurna
    [0.45, 0.22, 0.40, 0.48]   # Sampel 5: Lewat Matang
])

# Label Target 3 Kelas One-Hot: [Mentah, Matang, Lewat Matang]
Y_batch = np.array([
    [1, 0, 0],
    [1, 0, 0],
    [0, 1, 0],
    [0, 1, 0],
    [0, 0, 1]
])

# Buat model MLP dengan 4 fitur input -> 8 hidden -> 6 hidden -> 3 output
mlp_agro = BatchForwardMLP(layer_dims=[4, 8, 6, 3], seed=10)
probs_pred, cache_mlp = mlp_agro.forward(X_batch)

# Hitung Loss CCE
loss_batch = compute_log_sum_exp_cce(cache_mlp['Z3'], Y_batch)

print("=== HASIL EKSEKUSI FORWARD PROPAGATION BATCH ===")
print("Matriks Probabilitas Prediksi Luaran Softmax (5 Sampel x 3 Kelas):")
print(np.round(probs_pred, 4))
print(f"\nNilai Kerugian Rata-Rata Batch (CCE Loss): {loss_batch:.4f}")
print("Status: Seluruh aktivasi intermediate (A0, Z1, A1, Z2, A2, Z3, A3) tersimpan sempurna di cache!")